# Task 2.4 — Multi-Agent Signal Consolidation

Consolidate per-factor sentiments into filing-level investment signals and validate
against 21-day excess returns.

## Architecture

```
┌──────────────────┐     ┌───────────────────┐     ┌──────────────────────┐     ┌────────────────────┐
│  Agent 1          │     │  Agent 2           │     │  Agent 3              │     │  Agent 4            │
│  Factor Sentiment │────▶│  Category          │────▶│  Sub-Sector Context   │────▶│  Signal             │
│  (load scored)    │     │  Aggregator        │     │  (key drivers/risks)  │     │  Consolidator       │
└──────────────────┘     └───────────────────┘     └──────────────────────┘     └────────────────────┘
```

| Input | Source |
|-------|--------|
| Scored factors | `../task_1/output/factors_scored/{TICKER}/*.json` |
| Filing returns | `../task_1/output/filing_returns.csv` |
| Reasoning ICs | `../task_1/output/reasoning_output.json` |
| Ticker mapping | `../task_1/ticker_mapping.json` |

| Output | Destination |
|--------|-------------|
| Filing signals | `output/filing_signals.jsonl` |
| Cohort vs return chart | Inline |

> **Runs on ACCRE** — no GPU required. All agents are computational/rule-based.

## Step 1: Setup — Imports, Config, Data Loading

In [ ]:
import json
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr, pearsonr

# ── Paths ────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
TASK1_DIR = (_cwd.parent / "task_1").resolve() if (_cwd.parent / "task_1").exists() else (_cwd / ".." / "task_1").resolve()

FACTORS_SCORED_DIR = TASK1_DIR / "output" / "factors_scored"
RETURNS_PATH = TASK1_DIR / "output" / "filing_returns.csv"
REASONING_PATH = TASK1_DIR / "output" / "reasoning_output.json"
TICKER_MAP_PATH = TASK1_DIR / "ticker_mapping.json"
OUTPUT_DIR = _cwd / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ────────────────────────────────────────────────────────────
LABEL_ORDER = ["very_negative", "negative", "neutral", "positive", "very_positive"]
LABEL_TO_SCORE = {
    "very_negative": -2,
    "negative": -1,
    "neutral": 0,
    "positive": 1,
    "very_positive": 2,
}
COHORT_THRESHOLDS = [
    ("very_negative", -np.inf, -0.6),
    ("negative",      -0.6,   -0.2),
    ("neutral",       -0.2,    0.2),
    ("positive",       0.2,    0.6),
    ("very_positive",  0.6,    np.inf),
]

# ── Load ticker mapping ─────────────────────────────────────────────────
with open(TICKER_MAP_PATH) as f:
    _ticker_map = json.load(f)

TICKER_SUBSECTOR: dict[str, str] = {}
for subsector, tickers in _ticker_map.items():
    for t in tickers:
        TICKER_SUBSECTOR[t] = subsector

# ── Load filing returns ──────────────────────────────────────────────────
returns_df = pd.read_csv(RETURNS_PATH)
returns_df["filing_date"] = pd.to_datetime(returns_df["filing_date"])
returns_df = returns_df[returns_df["status"] == "ok"].copy()
returns_df = returns_df.dropna(subset=["ret_21d_excess"])

# ── Load reasoning ICs ───────────────────────────────────────────────────
with open(REASONING_PATH) as f:
    reasoning_data = json.load(f)

ic_rows = []
for sector in reasoning_data["sectors"]:
    for item in reasoning_data["sector_reasoning"][sector].get("sentiment_return_link", []):
        ic_rows.append({
            "sector": sector,
            "factor_key": item["factor"],
            "IC": item["IC"],
            "hit_rate": item["hit_rate"],
        })
ic_df = pd.DataFrame(ic_rows)

# ── Logging ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("task_2_4")

# ── Summary ──────────────────────────────────────────────────────────────
print(f"Task 1 dir:          {TASK1_DIR}")
print(f"Factors scored dir:  {FACTORS_SCORED_DIR} (exists={FACTORS_SCORED_DIR.exists()})")
print(f"Returns:             {len(returns_df)} filings (status=ok)")
print(f"Return stats:        mean={returns_df['ret_21d_excess'].mean():.4f}, "
      f"std={returns_df['ret_21d_excess'].std():.4f}")
print(f"IC data:             {len(ic_df)} factor-sector pairs across {ic_df['sector'].nunique()} sectors")
print(f"Tickers:             {len(TICKER_SUBSECTOR)} across {len(_ticker_map)} sub-sectors")
print(f"\nIC summary:")
print(ic_df.to_string(index=False))

## Step 2: Agent 1 — Factor Sentiment Agent

Load all scored factors from `factors_scored/` into a flat DataFrame. Each factor has a
5-class sentiment label assigned by the base LLM in task 1.3.

> **Design note:** The original blueprint calls for Agent 1 to re-score all ~67K factors
> with the SFT model. At 0.15 it/s (measured in task_2_3), that's ~130 hours — impractical.
> We use existing scored factors; SFT improvement was demonstrated in task_2_3
> (macro F1 = 0.731, +0.127 over base).

In [ ]:
rows = []
skipped = 0
files_loaded = 0

for ticker_dir in sorted(FACTORS_SCORED_DIR.iterdir()):
    if not ticker_dir.is_dir():
        continue
    ticker = ticker_dir.name
    subsector = TICKER_SUBSECTOR.get(ticker, "unknown")

    for fpath in sorted(ticker_dir.glob("*_factors.json")):
        with open(fpath) as f:
            data = json.load(f)

        files_loaded += 1
        form = data.get("form", "")
        filing_date = data.get("filing_date", "")

        for fac in data.get("factors", []):
            sentiment = fac.get("sentiment")
            if not isinstance(sentiment, dict) or not sentiment.get("label"):
                skipped += 1
                continue

            evidence_parts = []
            for ev in fac.get("evidence", []):
                text = ev.get("text", "").strip()
                if text:
                    evidence_parts.append(text)
            evidence_text = " | ".join(evidence_parts) if evidence_parts else ""

            rows.append({
                "ticker": ticker,
                "sub_sector": subsector,
                "form": form,
                "filing_date": filing_date,
                "key": fac.get("key", ""),
                "category": fac.get("category", ""),
                "summary": fac.get("summary", ""),
                "evidence_text": evidence_text,
                "label": sentiment["label"],
                "rationale": sentiment.get("rationale", ""),
                "confidence": sentiment.get("confidence", 0.0),
            })

factors_df = pd.DataFrame(rows)
factors_df["filing_date"] = pd.to_datetime(factors_df["filing_date"])
factors_df["score"] = factors_df["label"].map(LABEL_TO_SCORE)

print(f"Files loaded:         {files_loaded}")
print(f"Total factors:        {len(factors_df)}")
print(f"Skipped (no label):   {skipped}")
print(f"Unique tickers:       {factors_df['ticker'].nunique()}")
print(f"Unique filings:       {factors_df.groupby(['ticker', 'filing_date']).ngroups}")
print(f"Unique categories:    {factors_df['category'].nunique()}")
print(f"Unique factor keys:   {factors_df['key'].nunique()}")

print(f"\nLabel distribution:")
label_dist = factors_df["label"].value_counts().reindex(LABEL_ORDER)
for lbl, cnt in label_dist.items():
    print(f"  {lbl:16s}: {cnt:6d} ({cnt/len(factors_df):.1%})")

print(f"\nLabel × Sub-sector cross-tab:")
cross = pd.crosstab(factors_df["sub_sector"], factors_df["label"])
cross = cross.reindex(columns=LABEL_ORDER, fill_value=0)
print(cross.to_string())

## Step 3: Agent 2 — Category Aggregator

For each filing, compute a **confidence-weighted average sentiment score** per category.
This reduces ~25 factors per filing into ~14 category-level scores.

In [ ]:
def confidence_weighted_mean(group: pd.DataFrame) -> float:
    """Weighted average of score by confidence within a group."""
    w = group["confidence"].values
    s = group["score"].values
    total_w = w.sum()
    if total_w == 0:
        return s.mean()
    return (s * w).sum() / total_w


# Group by (ticker, filing_date, category) → weighted avg score
cat_scores = (
    factors_df
    .groupby(["ticker", "filing_date", "category"])
    .apply(confidence_weighted_mean, include_groups=False)
    .rename("cat_score")
    .reset_index()
)

# Pivot: one row per (ticker, filing_date), one column per category
cat_pivot = cat_scores.pivot_table(
    index=["ticker", "filing_date"],
    columns="category",
    values="cat_score",
)
cat_pivot.columns.name = None

# Merge with returns
filing_df = cat_pivot.reset_index().merge(
    returns_df[["ticker", "filing_date", "form", "ret_21d_excess"]],
    on=["ticker", "filing_date"],
    how="inner",
)

# Add sub-sector
filing_df["sub_sector"] = filing_df["ticker"].map(TICKER_SUBSECTOR)

# Identify category columns
cat_cols = [c for c in cat_pivot.columns if c not in ("ticker", "filing_date")]

print(f"Filing DataFrame:     {filing_df.shape[0]} filings × {len(cat_cols)} categories")
print(f"Category columns:     {cat_cols}")
print(f"Matched with returns: {filing_df['ret_21d_excess'].notna().sum()}")
print(f"\nCategory coverage (fraction of filings with non-null score):")
coverage = filing_df[cat_cols].notna().mean().sort_values(ascending=False)
for cat, frac in coverage.items():
    print(f"  {cat:30s}: {frac:.1%}")

print(f"\nSample rows:")
filing_df[["ticker", "filing_date", "form", "sub_sector", "ret_21d_excess"] + cat_cols[:4]].head(5)

## Step 4: Agent 3 — Sub-Sector Context Agent

For each filing, identify **key drivers** (top 3 positive categories) and **risk flags**
(top 3 negative categories). Flag sector-specific categories when they appear as
top drivers or risks.

In [ ]:
SECTOR_SPECIFIC_CATS = {
    "airlines":              ["airlines_transport"],
    "defense":               ["defense_government"],
    "industrial_equipment":  ["industrial_infrastructure"],
    "general":               [],
}


def identify_drivers_and_risks(row: pd.Series) -> dict:
    """Rank categories by score; extract top-3 drivers and top-3 risks."""
    scores = {cat: row[cat] for cat in cat_cols if pd.notna(row[cat])}
    if not scores:
        return {"key_drivers": [], "risk_flags": [], "sector_context": ""}

    sorted_cats = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # Key drivers: top-3 with positive score
    key_drivers = [cat for cat, sc in sorted_cats if sc > 0][:3]

    # Risk flags: bottom-3 with negative score
    risk_flags = [cat for cat, sc in sorted_cats[::-1] if sc < 0][:3]

    # Sector-specific context
    sub_sector = row.get("sub_sector", "")
    spec_cats = SECTOR_SPECIFIC_CATS.get(sub_sector, [])
    context_parts = []
    for sc in spec_cats:
        if sc in [d for d in key_drivers]:
            context_parts.append(f"{sc} is a key driver")
        elif sc in [r for r in risk_flags]:
            context_parts.append(f"{sc} is a risk flag")
    sector_context = "; ".join(context_parts) if context_parts else ""

    return {
        "key_drivers": key_drivers,
        "risk_flags": risk_flags,
        "sector_context": sector_context,
    }


agent3_results = filing_df.apply(identify_drivers_and_risks, axis=1, result_type="expand")
filing_df["key_drivers"] = agent3_results["key_drivers"]
filing_df["risk_flags"] = agent3_results["risk_flags"]
filing_df["sector_context"] = agent3_results["sector_context"]

# Stats
n_with_drivers = filing_df["key_drivers"].apply(len).gt(0).sum()
n_with_risks = filing_df["risk_flags"].apply(len).gt(0).sum()
n_with_context = filing_df["sector_context"].str.len().gt(0).sum()

print(f"Filings with key drivers:   {n_with_drivers} / {len(filing_df)}")
print(f"Filings with risk flags:    {n_with_risks} / {len(filing_df)}")
print(f"Filings with sector context:{n_with_context} / {len(filing_df)}")

# Most common drivers and risks
from collections import Counter

all_drivers = [cat for cats in filing_df["key_drivers"] for cat in cats]
all_risks = [cat for cats in filing_df["risk_flags"] for cat in cats]

print(f"\nTop key drivers:")
for cat, cnt in Counter(all_drivers).most_common(5):
    print(f"  {cat:30s}: {cnt}")

print(f"\nTop risk flags:")
for cat, cnt in Counter(all_risks).most_common(5):
    print(f"  {cat:30s}: {cnt}")

## Step 5: Agent 4 — Signal Consolidator

Compute the final filing-level signal as an **IC-weighted average** of category scores.
Weights = |IC| per factor (from `reasoning_output.json`), aggregated to category level
by averaging factor ICs within each category. Categories without IC data get equal weight.

In [ ]:
# ── Build factor_key → category mapping from actual factor data ───────────
key_to_cat = factors_df.groupby("key")["category"].first().to_dict()

# ── Build per-sector category-level IC weights ───────────────────────────
# Map factor-level ICs to categories, then average |IC| per category per sector
ic_df["category"] = ic_df["factor_key"].map(key_to_cat)
ic_df["abs_IC"] = ic_df["IC"].abs()

sector_cat_ic = (
    ic_df.dropna(subset=["category"])
    .groupby(["sector", "category"])["abs_IC"]
    .mean()
    .reset_index()
    .rename(columns={"abs_IC": "weight"})
)

print("IC-derived category weights per sector:")
for sector in sorted(sector_cat_ic["sector"].unique()):
    subset = sector_cat_ic[sector_cat_ic["sector"] == sector]
    print(f"\n  {sector}:")
    for _, r in subset.iterrows():
        print(f"    {r['category']:30s}: {r['weight']:.4f}")

# Default weight for categories without IC data
DEFAULT_WEIGHT = 1.0 / len(cat_cols)

# ── Compute IC-weighted signal per filing ─────────────────────────────────
def compute_signal(row: pd.Series) -> float:
    """IC-weighted sum of category scores, normalized to [-1, 1]."""
    sub_sector = row.get("sub_sector", "")
    # Get sector-specific weights
    sector_weights = sector_cat_ic[sector_cat_ic["sector"] == sub_sector]
    weight_map = dict(zip(sector_weights["category"], sector_weights["weight"]))

    weighted_sum = 0.0
    total_weight = 0.0

    for cat in cat_cols:
        val = row[cat]
        if pd.isna(val):
            continue
        w = weight_map.get(cat, DEFAULT_WEIGHT)
        weighted_sum += val * w
        total_weight += w

    if total_weight == 0:
        return 0.0
    return weighted_sum / total_weight


filing_df["signal"] = filing_df.apply(compute_signal, axis=1)

# Normalize to [-1, 1] via clipping at the score range extremes
# Max possible weighted avg is 2.0 (all very_positive), min is -2.0
filing_df["signal"] = filing_df["signal"].clip(-2, 2) / 2.0

# ── Assign cohort ────────────────────────────────────────────────────────
def assign_cohort(signal: float) -> str:
    for name, lo, hi in COHORT_THRESHOLDS:
        if lo <= signal < hi:
            return name
    return "very_positive"  # edge case: signal == 1.0


filing_df["cohort"] = filing_df["signal"].apply(assign_cohort)

# Enforce categorical ordering
filing_df["cohort"] = pd.Categorical(filing_df["cohort"], categories=LABEL_ORDER, ordered=True)

print(f"\nSignal statistics:")
print(f"  mean:   {filing_df['signal'].mean():.4f}")
print(f"  std:    {filing_df['signal'].std():.4f}")
print(f"  min:    {filing_df['signal'].min():.4f}")
print(f"  max:    {filing_df['signal'].max():.4f}")
print(f"  median: {filing_df['signal'].median():.4f}")

print(f"\nCohort distribution:")
cohort_counts = filing_df["cohort"].value_counts().reindex(LABEL_ORDER)
for cohort, cnt in cohort_counts.items():
    print(f"  {cohort:16s}: {cnt:5d} ({cnt/len(filing_df):.1%})")

## Step 6: Save Filing Signals

Write one JSON line per filing to `output/filing_signals.jsonl`.

In [ ]:
signals_path = OUTPUT_DIR / "filing_signals.jsonl"

with open(signals_path, "w") as f:
    for _, row in filing_df.iterrows():
        # Build category scores dict (non-null only)
        category_scores = {
            cat: round(float(row[cat]), 4)
            for cat in cat_cols
            if pd.notna(row[cat])
        }

        record = {
            "ticker": row["ticker"],
            "form": row["form"],
            "filing_date": row["filing_date"].strftime("%Y-%m-%d"),
            "sub_sector": row["sub_sector"],
            "signal": round(float(row["signal"]), 4),
            "cohort": row["cohort"],
            "category_scores": category_scores,
            "key_drivers": row["key_drivers"],
            "risk_flags": row["risk_flags"],
            "ret_21d_excess": round(float(row["ret_21d_excess"]), 6),
        }
        f.write(json.dumps(record) + "\n")

file_size = signals_path.stat().st_size
print(f"Saved: {signals_path}")
print(f"Rows:  {len(filing_df)}")
print(f"Size:  {file_size / 1024:.1f} KB")

## Step 7: Cohort Distribution

In [ ]:
sns.set_style("whitegrid")
cohort_colors = ["#d62728", "#ff7f0e", "#999999", "#2ca02c", "#1f77b4"]

fig, ax = plt.subplots(figsize=(10, 5))
counts = filing_df["cohort"].value_counts().reindex(LABEL_ORDER)
bars = ax.bar(range(len(LABEL_ORDER)), counts.values, color=cohort_colors)

ax.set_xticks(range(len(LABEL_ORDER)))
ax.set_xticklabels([l.replace("_", "\n") for l in LABEL_ORDER], fontsize=10)
ax.set_ylabel("Number of Filings", fontsize=11)
ax.set_title(f"Cohort Distribution — {len(filing_df)} Filings", fontsize=13)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            str(val), ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

## Step 8: Cohort vs Excess Return (Key Validation)

The central test: do positive-sentiment cohorts earn higher 21-day excess returns
than negative ones? A monotonic or near-monotonic pattern validates the pipeline.

In [ ]:
cohort_stats = (
    filing_df.groupby("cohort", observed=False)["ret_21d_excess"]
    .agg(["mean", "std", "count", "sem"])
    .reindex(LABEL_ORDER)
)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(LABEL_ORDER))
means = cohort_stats["mean"].values
sems = cohort_stats["sem"].values

bars = ax.bar(x, means * 100, yerr=sems * 100, capsize=5,
              color=cohort_colors, edgecolor="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels([l.replace("_", "\n") for l in LABEL_ORDER], fontsize=10)
ax.set_ylabel("Mean 21-Day Excess Return (%)", fontsize=11)
ax.set_title("Cohort vs. Mean 21-Day Excess Return", fontsize=13)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

for bar, m, s, n in zip(bars, means, sems, cohort_stats["count"].values):
    label_y = bar.get_height() + s * 100 + 0.05 if m >= 0 else bar.get_height() - s * 100 - 0.15
    ax.text(bar.get_x() + bar.get_width() / 2, label_y,
            f"{m*100:.2f}%\n(n={n})", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

# Spearman: cohort rank vs return
cohort_rank_map = {c: i for i, c in enumerate(LABEL_ORDER)}
filing_df["cohort_rank"] = filing_df["cohort"].map(cohort_rank_map)
rho, pval = spearmanr(filing_df["cohort_rank"], filing_df["ret_21d_excess"])
print(f"\nSpearman(cohort_rank, ret_21d_excess): rho={rho:.4f}, p={pval:.4e}")
print(f"\nCohort summary table:")
print(cohort_stats.to_string(float_format="{:.4f}".format))

## Step 9: Signal vs Return Scatter

Continuous signal vs 21-day excess return with regression line, colored by sub-sector.

In [ ]:
sector_palette = {
    "airlines": "#e41a1c",
    "defense": "#377eb8",
    "industrial_equipment": "#4daf4a",
    "general": "#984ea3",
}

fig, ax = plt.subplots(figsize=(10, 7))

for sector, color in sector_palette.items():
    mask = filing_df["sub_sector"] == sector
    ax.scatter(
        filing_df.loc[mask, "signal"],
        filing_df.loc[mask, "ret_21d_excess"] * 100,
        alpha=0.3, s=15, c=color, label=sector,
    )

# Regression line
z = np.polyfit(filing_df["signal"], filing_df["ret_21d_excess"] * 100, 1)
p = np.poly1d(z)
x_range = np.linspace(filing_df["signal"].min(), filing_df["signal"].max(), 100)
ax.plot(x_range, p(x_range), "k--", linewidth=2, label="OLS fit")

ax.set_xlabel("Filing Signal", fontsize=11)
ax.set_ylabel("21-Day Excess Return (%)", fontsize=11)
ax.set_title("Signal vs. 21-Day Excess Return", fontsize=13)
ax.legend(fontsize=9)
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)

plt.tight_layout()
plt.show()

# Correlations
rho_s, p_s = spearmanr(filing_df["signal"], filing_df["ret_21d_excess"])
rho_p, p_p = pearsonr(filing_df["signal"], filing_df["ret_21d_excess"])
print(f"Spearman: rho={rho_s:.4f}, p={p_s:.4e}")
print(f"Pearson:  r={rho_p:.4f}, p={p_p:.4e}")

## Step 10: Per-Sector Cohort Analysis

Break down cohort vs return by sub-sector. Identify which sub-sectors exhibit
monotonic cohort-return relationships.

In [ ]:
sectors = sorted(filing_df["sub_sector"].unique())
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
axes = axes.flatten()

for i, sector in enumerate(sectors):
    ax = axes[i]
    sector_data = filing_df[filing_df["sub_sector"] == sector]

    sector_cohort = (
        sector_data.groupby("cohort", observed=False)["ret_21d_excess"]
        .agg(["mean", "sem", "count"])
        .reindex(LABEL_ORDER)
    )

    x = np.arange(len(LABEL_ORDER))
    means = sector_cohort["mean"].fillna(0).values
    sems = sector_cohort["sem"].fillna(0).values
    counts = sector_cohort["count"].fillna(0).astype(int).values

    bars = ax.bar(x, means * 100, yerr=sems * 100, capsize=4,
                  color=cohort_colors, edgecolor="black", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([l.replace("_", "\n") for l in LABEL_ORDER], fontsize=8)
    ax.set_title(f"{sector} (n={len(sector_data)})", fontsize=11)
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")

    for bar, n in zip(bars, counts):
        if n > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, 0.02,
                    f"n={n}", ha="center", va="bottom", fontsize=7, color="gray")

    # Check monotonicity
    valid = [(m, c) for m, c in zip(means, counts) if c > 0]
    if len(valid) >= 3:
        valid_means = [v[0] for v in valid]
        is_monotonic = all(a <= b for a, b in zip(valid_means, valid_means[1:]))
        rho_sec, p_sec = spearmanr(
            sector_data["cohort_rank"], sector_data["ret_21d_excess"]
        )
        mono_label = "monotonic" if is_monotonic else "non-monotonic"
        ax.text(0.02, 0.95, f"Spearman={rho_sec:.3f} (p={p_sec:.3f})\n{mono_label}",
                transform=ax.transAxes, fontsize=8, va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

fig.supylabel("Mean 21-Day Excess Return (%)", fontsize=11)
fig.suptitle("Cohort vs. Return by Sub-Sector", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Step 11: Summary Statistics

In [ ]:
print("=" * 65)
print("  MULTI-AGENT SIGNAL CONSOLIDATION — SUMMARY")
print("=" * 65)

# Cohort table
print("\nCohort counts and mean returns:")
print("-" * 55)
print(f"{'Cohort':>16s}  {'Count':>6s}  {'Mean Ret%':>10s}  {'Std%':>8s}")
print("-" * 55)
for cohort in LABEL_ORDER:
    mask = filing_df["cohort"] == cohort
    n = mask.sum()
    if n > 0:
        m = filing_df.loc[mask, "ret_21d_excess"].mean() * 100
        s = filing_df.loc[mask, "ret_21d_excess"].std() * 100
        print(f"{cohort:>16s}  {n:6d}  {m:10.3f}  {s:8.3f}")
    else:
        print(f"{cohort:>16s}  {n:6d}       N/A       N/A")

# Key metrics
rho_signal, p_signal = spearmanr(filing_df["signal"], filing_df["ret_21d_excess"])
rho_cohort, p_cohort = spearmanr(filing_df["cohort_rank"], filing_df["ret_21d_excess"])

# Long-short return
vp_mask = filing_df["cohort"] == "very_positive"
vn_mask = filing_df["cohort"] == "very_negative"
long_ret = filing_df.loc[vp_mask, "ret_21d_excess"].mean() if vp_mask.any() else np.nan
short_ret = filing_df.loc[vn_mask, "ret_21d_excess"].mean() if vn_mask.any() else np.nan
ls_return = long_ret - short_ret if not (np.isnan(long_ret) or np.isnan(short_ret)) else np.nan

# Information ratio
mean_excess = filing_df["ret_21d_excess"].mean()
std_excess = filing_df["ret_21d_excess"].std()
ir = mean_excess / std_excess if std_excess > 0 else np.nan

print(f"\n{'Key Metrics':}")
print(f"  Spearman(signal, return):     {rho_signal:.4f} (p={p_signal:.4e})")
print(f"  Spearman(cohort, return):     {rho_cohort:.4f} (p={p_cohort:.4e})")
print(f"  Long-short return (VP - VN):  {ls_return*100:.3f}%")
print(f"  Information ratio:            {ir:.4f}")
print(f"  Total filings:                {len(filing_df)}")
print(f"  Total scored factors:         {len(factors_df)}")
print(f"  Output file:                  {signals_path}")

print("\n" + "=" * 65)